# About Dataset

Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

* asin - ID of the product, like B000FA64PK
* helpful - helpfulness rating of the review - example: 2/3.
* overall - rating of the product.
* reviewText - text of the review (heading).
* reviewTime - time of the review (raw).
* reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
* reviewerName - name of the reviewer.
* summary - summary of the review (description).
* unixReviewTime - unix timestamp.
Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration

* Sentiment analysis on reviews.
* Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
* Fake reviews/ outliers.
* Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
* Any other interesting analysis

## Load the dataset

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

data = pd.read_csv('all_kindle_review.csv')
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [2]:
df=data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [3]:
df.shape

(12000, 2)

In [4]:
## Missing Values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [5]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [6]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

## Preprocessing And Cleaning

In [7]:
## postive review is 1 and negative review is 0
df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)

In [8]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [9]:
## 1. Lower All the cases
df['reviewText']=df['reviewText'].str.lower()

In [10]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [11]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\priye\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from bs4 import BeautifulSoup

In [13]:
## Removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url 
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [14]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [15]:
## Lemmatizer
from nltk.stem import WordNetLemmatizer

lemmatizer=WordNetLemmatizer()

In [16]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [17]:
df['reviewText']=df['reviewText'].apply(lambda x:lemmatize_words(x))

In [18]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [39]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'], test_size=0.20, random_state=42)

## Bag Of Words

In [64]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()

In [65]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 35736))

In [66]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)

In [ ]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report, precision_score, recall_score

y_pred_bow=nb_model_bow.predict(X_test_bow)

In [68]:
confusion_matrix(y_test,y_pred_bow)

array([[499, 304],
       [717, 880]])

In [69]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))
print("BOW precision: ",precision_score(y_test,y_pred_bow))
print("BOW recall: ",recall_score(y_test,y_pred_bow))

BOW accuracy:  0.5745833333333333
BOW precision:  0.7432432432432432
BOW recall:  0.5510331872260489


## TF-IDF

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

In [71]:
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [72]:
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)

In [73]:
confusion_matrix(y_test,y_pred_tfidf)

array([[488, 315],
       [696, 901]])

In [74]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))
print("TFIDF precision: ",precision_score(y_test,y_pred_tfidf))
print("TFIDF recall: ",recall_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.57875
TFIDF precision:  0.740953947368421
TFIDF recall:  0.5641828428303068


## Word2Vec

In [80]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [81]:
words = []
for sent in df['reviewText']:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [82]:
import gensim 
import numpy as np

model = gensim.models.Word2Vec(words)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [83]:
def avg_word2vec(doc):
    ## remove out-of-vocabulary words
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)

In [84]:
from tqdm import tqdm

## apply for the entire sentences 
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

100%|██████████| 12000/12000 [00:16<00:00, 740.44it/s]


In [85]:
## Dependent Feature
y = df[list(map(lambda x: len(x)>0 ,df['reviewText']))]
y=pd.get_dummies(y['rating'])
y=y.iloc[:,0].values

In [86]:
## this is the final independent features
df = pd.DataFrame()

for i in range(len(X)):
    temp_df = pd.DataFrame(X[i].reshape(1, -1))
    df = pd.concat([df, temp_df], ignore_index=True)

In [87]:
df['Output']=y

In [88]:
df.isna().sum()

0         0
1         0
2         0
3         0
4         0
         ..
96        0
97        0
98        0
99        0
Output    0
Length: 101, dtype: int64

In [89]:
df.dropna(inplace=True)

In [90]:
## Independent Feature 
X = df.iloc[:, :-1]

## Dependent Feature
y=df['Output']

In [91]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [92]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier(random_state=42)

rf_model = rf_clf.fit(X_train, y_train)

In [93]:
## Evaluate the Model
from sklearn.metrics import accuracy_score, classification_report
y_pred = rf_model.predict(X_test)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred)}")
print(f"Precision Score: {precision_score(y_test, y_pred)}")
print(f"Recall Score: {recall_score(y_test, y_pred)}")

Accuracy Score: 0.7654166666666666
Precision Score: 0.6916932907348243
Recall Score: 0.539227895392279


In [94]:
print(confusion_matrix(y_test, y_pred))

[[1404  193]
 [ 370  433]]
